# Employee Attrition & Retention Analysis — EDA & Modeling

**Dataset:** IBM HR Analytics Employee Attrition (1,470 employees)  
**Data:** `data/processed/employee_attrition_clean.csv`  
**Models:** Logistic Regression baseline + XGBoost classifier

This notebook mirrors `src/model_training.py`. Prefer running the script for the exported risk-score CSV.

## 1. Load & Inspect Data

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

DATA_PATH = ROOT / 'data' / 'processed' / 'employee_attrition_clean.csv'
RISK_PATH = ROOT / 'data' / 'processed' / 'employee_risk_scores.csv'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
print(df.info())
display(df.describe(include='number').T.head(10))
print('\nAttrition class balance:')
print(df['attrition'].value_counts())
print(df['attrition'].value_counts(normalize=True).round(4))

## 2. Exploratory Data Analysis

- Overall attrition rate
- Overtime × attrition
- Tenure and income distributions

In [ ]:
overall_rate = (df['attrition'] == 'Yes').mean()
print(f'Overall attrition rate: {overall_rate:.2%} ({(df.attrition == "Yes").sum()} / {len(df)})')

ot = (
    df.groupby('over_time')['attrition']
      .apply(lambda s: (s == 'Yes').mean())
      .rename('attrition_rate')
      .reset_index()
)
ot['headcount'] = df.groupby('over_time').size().values
print('\nAttrition by overtime:')
display(ot)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=df, x='over_time', hue='attrition', ax=axes[0])
axes[0].set_title('Attrition by OverTime')
sns.barplot(data=ot, x='over_time', y='attrition_rate', ax=axes[1], color='steelblue')
axes[1].set_title('Attrition rate by OverTime')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.tight_layout()
plt.show()

In [ ]:
df = df.copy()
df['tenure_band'] = pd.cut(
    df['years_at_company'],
    bins=[-0.1, 2, 5, 10, df['years_at_company'].max()],
    labels=['0-2 yrs', '2-5 yrs', '5-10 yrs', '10+ yrs'],
)

tenure = (
    df.groupby('tenure_band', observed=True)
      .agg(headcount=('attrition', 'size'),
           attrition_rate=('attrition', lambda s: (s == 'Yes').mean()))
      .reset_index()
)
print('Attrition by tenure band:')
display(tenure)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(data=df, x='years_at_company', hue='attrition', bins=20, ax=axes[0], element='step')
axes[0].set_title('Tenure distribution by attrition')
sns.histplot(data=df, x='monthly_income', hue='attrition', bins=25, ax=axes[1], element='step')
axes[1].set_title('Monthly income distribution by attrition')
plt.tight_layout()
plt.show()

## 3. Feature Engineering & Preprocessing

- One-hot encode categoricals; scale numerics for logistic regression
- Stratified train/test split (class imbalance ~16%)

In [ ]:
from model_training import (
    prepare_xy,
    train_baseline_model,
    train_xgboost_model,
    evaluate_model,
    get_feature_importance,
    export_risk_scores,
    build_preprocessor,
    overtime_rank,
    RANDOM_STATE,
    TEST_SIZE,
)
from sklearn.model_selection import train_test_split

model_df = pd.read_csv(DATA_PATH)
X, y, ids = prepare_xy(model_df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f'Train/test: {len(X_train)} / {len(X_test)}')
print(f'Train positive rate: {y_train.mean():.2%} | Test: {y_test.mean():.2%}')

## 4. Modeling: Logistic Regression Baseline

In [ ]:
lr_preprocessor = build_preprocessor(X_train)
lr_model = train_baseline_model(X_train, y_train, lr_preprocessor)
lr_metrics = evaluate_model(lr_model, X_test, y_test, name='Logistic Regression')

lr_importance = get_feature_importance(lr_model, top_n=15)
print('Top |coefficients|:')
display(lr_importance)
print('OverTime rank:', overtime_rank(lr_importance, 'abs_coefficient'))

## 5. Modeling: XGBoost Classifier

In [ ]:
xgb_model = train_xgboost_model(X_train, y_train, lr_preprocessor)
xgb_metrics = evaluate_model(xgb_model, X_test, y_test, name='XGBoost')

xgb_importance = get_feature_importance(xgb_model, top_n=15)
print('Top feature importances:')
display(xgb_importance)
print('OverTime rank:', overtime_rank(xgb_importance, 'importance'))

## 6. Feature Importance

OverTime is expected among the top drivers for both models (matches SQL EDA: ~30% attrition with overtime vs ~10% without).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=lr_importance, y='feature', x='abs_coefficient', ax=axes[0], color='steelblue')
axes[0].set_title('Logistic Regression — |coefficient|')
sns.barplot(data=xgb_importance, y='feature', x='importance', ax=axes[1], color='darkorange')
axes[1].set_title('XGBoost — feature importance')
plt.tight_layout()
plt.show()

print('Metric summary')
print(f"  Logistic  ROC-AUC={lr_metrics['roc_auc']:.3f}  P={lr_metrics['precision']:.3f}  R={lr_metrics['recall']:.3f}")
print(f"  XGBoost   ROC-AUC={xgb_metrics['roc_auc']:.3f}  P={xgb_metrics['precision']:.3f}  R={xgb_metrics['recall']:.3f}")

## 7. Export Risk Scores

Score all employees with XGBoost and write `data/processed/employee_risk_scores.csv`  
(`employee_number`, `attrition_probability`, `risk_tier`).

Cost-of-turnover modeling is in Commit 4 (Excel).

In [ ]:
risk_scores = export_risk_scores(xgb_model, X, ids, RISK_PATH)
print(f'Wrote {len(risk_scores)} rows to {RISK_PATH}')
display(risk_scores['risk_tier'].value_counts())
display(risk_scores.head(10))